# Import 

In [ ]:
# Week 8: Dựa vào job_dataset.ods & kiến thức đã học để cải thiện mô hình classification

# Import libraries:
!pip install pandas odfpy
from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Import dataset:
job_df = pd.read_excel(f'/Users/ngocta/Desktop/CoderSchool-AI/Week8/job_dataset.ods', engine='odf', dtype = str)
target = 'career_level'

# Checking data statistics:
#job_profile = ProfileReport(job_df, title = "Job Report", explorative = True)
#job_profile.to_file("Job Report.html")

# Remove empty row (only 1 row with missing data):
clean_job_df = job_df.dropna()

# Remove duplicate rows:
clean_job_df = clean_job_df.drop_duplicates()

# Remove misplellings in Description column:
clean_job_df['description'] = (
    clean_job_df['description']
    .str.replace("Â ", "", regex=False)
    .str.replace("Â  ", " -", regex=False)
    .str.replace("â€™", "'", regex=False)
    .str.replace("&amp;", "&", regex=False)
    .str.replace("â€¢", " -", regex=False)
)

# Clean Location column:
def filter_location(location):
    if ", " in location:
        return location[-2:]
    else:
        return location

clean_job_df['location'] = clean_job_df['location'].apply(filter_location)

# Split to train & test sets:
x = clean_job_df.drop(columns = target, axis = 1)
y = clean_job_df[target]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 100, stratify = y)

# OVERSAMPLING MINORITY CLASS IN CAREER LEVEL COLUMN:
train_data = x_train.copy()
train_data['career_level'] = y_train

print(clean_job_df['career_level'].value_counts())
minority_classes = [
    "managing_director_small_medium_company",
    "specialist",
    "director_business_unit_leader"
]

# Duplicate rows of minority classes
minority_data = train_data[train_data['career_level'].isin(minority_classes)]
duplicated_data = pd.concat([minority_data] * 10, ignore_index=True)

# Combine original data with duplicated data:
balanced_data = pd.concat([train_data, duplicated_data], ignore_index=True)

# Update x_train & y_train:
x_train = balanced_data.drop(columns='career_level')
y_train = balanced_data['career_level']
print(balanced_data['career_level'].value_counts())

# TEST DIFFERENT TFIDFVECTORIZER NGRAM FOR DESCRIPTION COLUMN:
ngrams = [(1, 1), (1, 2), (1, 3)]

for ngram in ngrams:
    test_preprocessor = ColumnTransformer(transformers=[
        ('title', TfidfVectorizer(ngram_range = ngram, stop_words = 'english'), 'title'),
        ('location', OneHotEncoder(handle_unknown='ignore'), ['location']),
        ('description', TfidfVectorizer(ngram_range = ngram, stop_words = 'english'), 'description'),
        ('function', OneHotEncoder(handle_unknown = 'ignore'), ['function']),
        ('industry', TfidfVectorizer(ngram_range = ngram, stop_words = 'english'), 'industry')
    ])
    test_output = test_preprocessor.fit_transform(x_train)
    print(f"Preprocessor using ngram_range: {ngram}")
    print(test_output.shape)
# CONCLUSION: best ngram = (1,2) as it improve understanding but still efficient to run    

# Vectorize text columns:
preprocessor = ColumnTransformer(transformers = [
    ('title', TfidfVectorizer(ngram_range = (1,2), stop_words = 'english'), 'title'),
    ('location', OneHotEncoder(handle_unknown ='ignore'), ['location']),
    ('description', TfidfVectorizer(ngram_range = (1,2), stop_words = 'english'), 'description'),
    ('function', OneHotEncoder(handle_unknown = 'ignore'), ['function']),
    ('industry', TfidfVectorizer(ngram_range = (1,2), stop_words = 'english'), 'industry')
])

output = preprocessor.fit_transform(x_train)
print(output.shape)

#vectorizer = TfidfVectorizer()
#output = vectorizer.fit_transform(x_train['title'])
#print(output.shape)
#print(vectorizer.vocabulary_)
#print(len(vectorizer.vocabulary_))

In [ ]:
# FIND THE OPTIMAL MODEL:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(),
    'SVC': SVC(),
    'Decision Tree': DecisionTreeClassifier(),
    'Multinomial NB': MultinomialNB()  
}

for name, item in models.items():
    print(f"\n{name}:")
    model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', item)
    ])
    
    # Train the model:
    model.fit(x_train, y_train)
    
    # Predict using the model:
    y_predict = model.predict(x_test)
    
    # Print classification report:
    print(classification_report(y_test, y_predict))

# CONCLUSION: most optimal model is Logistic Regression

In [ ]:
# OPTIMIZE LOGISTIC REGRESSION MODEL: 
# Define parameters (must support multiple classes)
LR_model = LogisticRegression(
    class_weight = 'balanced', 
    max_iter = 3000,
    solver = 'liblinear'
)

# Build the full pipelines:
final_model = Pipeline(steps = [
    ('Preprocessor', preprocessor),
    ('Model', LR_model)
])

# Train the model
final_model.fit(x_train, y_train)

# Predict and evaluate
y_predict = final_model.predict(x_test)
print(y_predict)
print("JOB REPORT:")
print(classification_report(y_test, y_predict, zero_division = 0))

# CONCLUSION: when comparing data with and without oversampling, it seemed recall for class specialist 
# & bereichtsleiter improved while F1 for class director_business_unit_leader dropped and 
# managing_director_small_medium_company unchanged because the samples of these 2 classes are still rare after 
# oversampling. However, when increasing duplication factor for Oversampling (100 vs 10), precision for 
# minority classes continue to drop while recall doesn't further improve. 
# So it's better to stick with lower duplication factor (10)